In [1]:
# 配置环境变量
from dotenv import load_dotenv
load_dotenv()
import os
import json
GROQ_API_KEY = os.getenv("GROQ_API_KEY")
if not GROQ_API_KEY or GROQ_API_KEY == "your_groq_api_key_here":
    raise ValueError(
        "\n请先在 .env 文件中设置有效的 GROQ_API_KEY\n"
        "访问 https://console.groq.com/keys 获取免费密钥"
    )

from langchain.chat_models import init_chat_model
model = init_chat_model(
    model="groq:llama-3.3-70b-versatile", api_key=GROQ_API_KEY
)

from pydantic import BaseModel, Field, field_validator, ValidationError
from enum import Enum
from typing import List, Optional, TypeVar, Type
# 泛型类型
T = TypeVar("T", bound=BaseModel)

In [2]:
class ExtractedData(BaseModel):
    """提取的数据（完整验证）"""
    name: str = Field(description="名称（字符串类型）", min_length=1)
    value: float = Field(description="数值（数字类型，必须 》 0）", gt=0)

    @field_validator("name")
    @classmethod
    def validate_name(cls, v):
        if v.strip() == "":
            raise ValueError("名称不能为空")
        return v.strip()

In [3]:
def extract_with_validation(text: str, max_retries: int = 3) -> Optional[ExtractedData]:
    """
    带验证的提取函数

    Args:
        text: 待提取的文本
        max_retries: 最大重试次数

    Returns:
        提取的数据（验证通过）或 None（失败）
    """
    structured_llm = model.with_structured_output(ExtractedData)

    current_text = text

    for attempt in range(1, max_retries + 1):
        try:
            # 调用 LLM（强调类型）
            prompt = f"""提取以下文本中的信息。
重要：value 必须是数字类型（float），不能是字符串。

{current_text}"""
            result = structured_llm.invoke(prompt)

            # 额外的业务验证（Pydantic 已经检查了 gt=0）
            # 所有验证通过
            return result

        except ValidationError as e:
            error_msg = e.errors()[0]['msg']
            if attempt < max_retries:
                current_text = f"{text}\n\n注意：{error_msg}。请重新提取。"
            else:
                return None

        except Exception as e:
            # 捕获 API 错误
            if attempt < max_retries:
                current_text = f"{text}\n\n重要：value 必须是数字类型，不能是字符串。"
            else:
                return None

    return None

In [4]:
def example():
    # 先创建结构化输出
    structured_primary = model.with_structured_output(ExtractedData)

    # 配置备用模型
    fallback_model = init_chat_model("groq:llama-3.3-70b-versatile", api_key=GROQ_API_KEY)
    structured_fallback = fallback_model.with_structured_output(ExtractedData)

    # 添加重试
    primary_with_retry = structured_primary.with_retry(
        retry_if_exception_type=(ConnectionError, TimeoutError),
        stop_after_attempt=2
    )

    # 降级
    robust_llm = primary_with_retry.with_fallbacks([structured_fallback])

    try:
        prompt = """提取以下文本中的信息。
重要：value 必须是数字类型（float）。

产品 C 的价值是 1299 元"""
        result = robust_llm.invoke(prompt)
        print(f"\n✓ 成功提取:")
        print(f"  名称: {result.name}")
        print(f"  数值: {result.value}")
    except Exception as e:
        print(f"\n✗ 所有策略都失败: {e}")

In [5]:
def main():
    print("\n"+"="*70)
    print("validation_retry")
    print("="*70)

    example()

In [6]:
if __name__ == "__main__":
    main()


validation_retry

✓ 成功提取:
  名称: 产品 C 的价值
  数值: 1299.0
